## Exercises on Generalizing with Neural Networks

These paper-and-pencil exercises reinforce Chapter 07. Exercises 7.1–7.7 cover the first part of the chapter (function approximation and Neural Fitted Q): why tabular methods fail in large or continuous state spaces, value functions as parametric approximators $Q_\theta(s,a)$, the mean-squared TD loss, the **semi-gradient** update (treating the bootstrapped target as a constant), the mini-batch loss, the handling of terminal and truncated transitions, the *fitted* (fixed-target) iteration and the non-stationary-target problem. Exercises 7.8–7.13 cover the second part (DQN and its refinements, each on the environment where the chapter tests it): the decaying $\varepsilon$ schedule, the replay buffer, the two targets of DQN and Double DQN, prioritised experience replay (priorities, sampling probabilities, importance-sampling weights) and the dueling aggregation with its identifiability constraint. Notation follows the notes: $\theta$ are the approximator parameters, with $\theta_{\text{online}}$ and $\theta_{\text{target}}$ for the two networks of DQN; $\phi(s,a)$ is the feature vector of a linear approximator; $y$ is the bootstrapped target, $\delta = y - Q$ the TD error, $\alpha$ the learning rate, $\gamma$ the discount and $d$ the terminal flag (the `failure` field of the transition record, equal to `terminated` and never to `truncated`).

### Exercise 7.1 — The cost of discretising a continuous state space

The Cart-Pole state has $n=4$ continuous variables and $|\mathcal{A}|=2$ actions. Suppose we discretise each state variable into $b$ bins and store a tabular $Q$.

1. How many state cells and how many $Q$-table entries result, as a function of $b$?
2. Evaluate for $b=10$ and $b=20$.
3. Two of the four variables (the velocities) are unbounded — comment on the additional difficulty.

**Step 1 — Count.** With $b$ bins per dimension and $n$ dimensions, the number of discrete states is $b^{n}$; the $Q$-table has one entry per (state, action):

$\displaystyle \#\text{cells}=b^{n}, \qquad \#\text{entries}=|\mathcal{A}|\,b^{n}.$

**Step 2 — Evaluate** ($n=4,\ |\mathcal{A}|=2$):

| $b$ | cells $b^4$ | entries $2b^4$ |
|---|---|---|
| $10$ | $10000$ | $20000$ |
| $20$ | $160000$ | $320000$ |

**Step 3 — Unbounded variables.** The cart and pole *velocities* range over $(-\infty,\infty)$, so any finite binning must first **clip** them to an arbitrary range; states outside the range collapse into edge bins, losing information. Finer binning (larger $b$) reduces this error but makes $b^n$ explode — the **curse of dimensionality**. Most cells are also visited rarely or never, so the table cannot be filled reliably.

**Key concept**

Tabular representation scales as $b^{n}$: exponential in the number of state variables, and impossible for continuous states without discretisation. Note that for $n=4$ memory is not the real obstacle (a few hundred thousand entries are nothing); the obstacles are the **time needed to visit every cell often enough** and the **absence of generalisation**: two adjacent cells share no information, so what is learned in one tells nothing about the other. This motivates **function approximation**, which *generalises* across nearby states instead of storing each one.

### Exercise 7.2 — A semi-gradient update with a linear approximator

Use a linear action-value approximator $Q_\theta(s,a)=\theta^\top\phi(s,a)$ with weights $\theta=(0.2,-0.1,0.5)$ and feature vector $\phi(s,a)=(1,0,2)$ for the current pair. A transition gives reward $r=1$, $\gamma=0.9$, and the greedy next-state value is $\max_{a'}Q_\theta(s',a')=2.0$. Learning rate $\alpha=0.1$.

1. Compute the current estimate $Q_\theta(s,a)$.
2. Form the (bootstrapped) TD target and the TD error $\delta$.
3. Perform one **semi-gradient** update $\theta\leftarrow\theta+\alpha\,\delta\,\phi(s,a)$ and recompute $Q_\theta(s,a)$.

**Step 1 — Current estimate.** $\ Q_\theta(s,a)=\theta^\top\phi = 0.2(1)+(-0.1)(0)+0.5(2) = 1.2.$

**Step 2 — TD target and error.** The target is treated as a fixed number (it is *detached* from $\theta$):

$\displaystyle y = r + \gamma\max_{a'}Q_\theta(s',a') = 1 + 0.9(2.0) = 2.8, \qquad \delta = y-Q_\theta(s,a) = 2.8-1.2 = 1.6.$

**Step 3 — Semi-gradient step.** For a linear model $\nabla_\theta Q_\theta(s,a)=\phi(s,a)$, so

$\displaystyle \theta \leftarrow \theta + \alpha\,\delta\,\phi = (0.2,-0.1,0.5) + 0.1(1.6)(1,0,2) = (0.36, -0.1, 0.82).$

New estimate: $\ Q_\theta(s,a)=\theta^\top\phi = 2$, which has moved from $1.2$ toward the target $2.8$ (by exactly $\alpha\delta\|\phi\|^2 = 0.1(1.6)(5)=0.8$).

**Key concept**

Function approximation replaces a table cell by a *weighted feature sum*; the update nudges the shared weights, so **every state that shares features with $(s,a)$ is affected** — this is generalisation. The gradient of a linear model is just its feature vector; for a neural network it comes from backpropagation, and the update has the same form.

### Exercise 7.3 — Why "semi-gradient"? (target treated as constant)

The NFQ loss for one transition is $\mathcal{L}(\theta)=\big(y-Q_\theta(s,a)\big)^2$, where the target $y=r+\gamma\max_{a'}Q_\theta(s',a')$ **also depends on $\theta$** (it is produced by the same network). Derive the gradient (i) treating $y$ as a constant (the semi-gradient used in practice), and (ii) differentiating $y$ as well (the full gradient), and explain why the semi-gradient is used.

**Step 1 — Semi-gradient (target detached).** Treating $y$ as a constant label:

$\displaystyle \nabla_\theta \mathcal{L} = -2\big(y - Q_\theta(s,a)\big)\,\nabla_\theta Q_\theta(s,a) = -2\,\delta\,\nabla_\theta Q_\theta(s,a).$

A gradient-descent step $\theta \leftarrow \theta - \tfrac{\alpha}{2}\nabla_\theta\mathcal{L}$ gives the update $\theta \leftarrow \theta + \alpha\,\delta\,\nabla_\theta Q_\theta(s,a)$ of Exercise 7.2: the factor $2$ is absorbed into the step size, which is why the notes write the update without it (some texts put a $\tfrac12$ in front of the loss for the same reason).

**Step 2 — Full gradient.** If we also differentiate $y=r+\gamma\max_{a'}Q_\theta(s',a')$ (assume the max is attained at $a'^*$):

$\displaystyle \nabla_\theta \mathcal{L} = -2\big(y-Q_\theta(s,a)\big)\Big[\nabla_\theta Q_\theta(s,a) - \gamma\,\nabla_\theta Q_\theta(s',a'^*)\Big].$

The extra term $+2\gamma\,\delta\,\nabla_\theta Q_\theta(s',a'^*)$ is what the semi-gradient **drops**.

**Step 3 — Why drop it.** The full gradient defines a legitimate algorithm, the *residual-gradient* method (Baird, 1995), which minimises the mean squared Bellman error. It is not used in practice for two reasons. First, when the transitions are stochastic the sampled full gradient is **biased**: the term $\nabla_\theta Q_\theta(s',a'^*)$ and the error $\delta$ are computed from the *same* sampled $s'$, so their product does not estimate the product of the expectations; an unbiased estimate would need two independent samples of $s'$ from the same $(s,a)$ (the *double-sampling problem*), which a real environment does not provide. Second, even in deterministic environments the method is much slower, because the second term pulls the value of $s'$ *toward* $Q_\theta(s,a)$ while the first pulls $Q_\theta(s,a)$ toward $s'$, and the two effects partially cancel. The semi-gradient avoids both issues by treating $y$ as a supervised label: each update is a step of an ordinary regression toward a fixed target, which is exactly the *fitted* viewpoint of NFQ, and exactly what `torch.no_grad()` does around the target in `optimize_nfq`. The price is that the semi-gradient is not the gradient of any fixed objective, so the convergence guarantees of gradient descent do not apply to it.

**Key concept**

In supervised learning the target is a fixed label; in bootstrapped RL the target is produced by the very parameters being trained. Pretending it is constant (the **semi-gradient**) is what makes the update well-behaved — and is *the* recurring subtlety of value-based deep RL.

### Exercise 7.4 — Mean-squared loss over a mini-batch

A mini-batch of three transitions has current predictions $Q_\theta(s_i,a_i)=(1.2,\,0.5,\,2.0)$ and (fixed, detached) targets $y_i=(2.8,\,0.5,\,1.0)$. Compute the mean-squared TD loss and state the sign of the update each sample induces.

**Step 1 — Per-sample TD errors** $\delta_i=y_i-Q_i$:

$\displaystyle \delta_1 = 2.8-1.2 = 1.6,\qquad \delta_2 = 0.5-0.5 = 0,\qquad \delta_3 = 1.0-2.0 = -1.0.$

**Step 2 — Mean-squared loss.**

$\displaystyle \mathcal{L} = \frac{1}{3}\sum_{i=1}^{3}\delta_i^2 = \frac{1.6^2 + 0^2 + (-1.0)^2}{3} = \frac{2.56 + 0 + 1.0}{3} = 1.1867.$

This is exactly what `(target - q_current).pow(2).mean()` computes in the notes.

**Step 3 — Update directions.** The semi-gradient step moves each prediction in the direction of its target (by exactly $\alpha\delta_i$ only in the tabular case, where $\nabla_\theta Q$ is a one-hot vector): sample 1 ($\delta_1>0$) is pushed **up**, sample 3 ($\delta_3<0$) is pushed **down**, and sample 2 ($\delta_2=0$) contributes nothing. Because the weights are shared, the net gradient is the average of the per-sample gradients.

**Key concept**

Mini-batch training averages the gradient over several transitions, reducing variance and smoothing the update — but the samples must be reasonably **independent** for this to behave like supervised learning, which sets up the correlation problem addressed by experience replay.

### Exercise 7.5 — Chasing a moving target (non-stationarity)

Consider a scalar linear approximator $Q_\theta(s)=\theta\,\phi(s)$ with a single feature. Two adjacent states share the *same* feature value $\phi(s)=\phi(s')=1$ (they look identical to the approximator). A transition $s\to s'$ gives $r=0$, $\gamma=0.9$; start with $\theta=1$ and $\alpha=0.5$.

1. Compute the TD target for this transition, and perform one semi-gradient update of $\theta$.
2. Recompute the target for the **same** transition after the update, and explain what happened.

**Step 1 — Target and update.** Current values: $Q_\theta(s)=\theta\phi(s)=1$, $Q_\theta(s')=\theta\phi(s')=1$.

$\displaystyle y = r + \gamma Q_\theta(s') = 0 + 0.9(1) = 0.9, \qquad \delta = 0.9 - 1 = -0.1.$
$\displaystyle \theta \leftarrow \theta + \alpha\,\delta\,\phi(s) = 1 + 0.5(-0.1)(1) = 0.95.$

**Step 2 — Recompute the target after the update.** Because $s$ and $s'$ share the feature, updating $\theta$ to fit $s$ also changed $Q_\theta(s')$:

$\displaystyle y' = r + \gamma Q_{\theta_{\text{new}}}(s') = 0.9\,(0.95)(1) = 0.855.$

The target moved from $0.9$ to $0.855$ — we stepped *toward* a target that then *shifted away*.

**Step 3 — Interpretation.** With a shared/generalising representation, an update aimed at $Q_\theta(s)$ perturbs $Q_\theta(s')$, so the bootstrapped target is **non-stationary**: it changes every time we train. This "chasing your own tail" is the mechanism behind the instability of naive value-based deep RL.

**Step 4 — Does this example diverge?** Iterating the update on the same transition gives

$\displaystyle \theta_{k+1} = \theta_k + \alpha\,(\gamma\theta_k - \theta_k) = \big(1 - \alpha(1-\gamma)\big)\,\theta_k = 0.95\,\theta_k \ \longrightarrow\ 0,$

so here the chase *converges*, and to the right value: with $r=0$ the true value of both states is $0$. Non-stationarity of the target is not by itself divergence. Divergence requires the full **deadly triad** (function approximation, bootstrapping, off-policy data); in this example the data is on-policy and the contraction factor $\gamma<1$ still wins. Baird's counterexample (Sutton & Barto, §11.2) shows that with off-policy updates the same kind of linear approximator can diverge to infinity.

**Key concept**

Function approximation couples states through shared parameters, making the learning target move as the weights move. In favourable cases the moving target still converges; in the off-policy setting of Q-learning it can diverge. Stabilising the target — with a **frozen target network** updated only occasionally — is precisely the fix introduced by DQN in the second part of this chapter.

### Exercise 7.6 — Terminal and truncated transitions

In the Cart-Pole every step gives $r=1$ and $\gamma=0.99$. For a transition $(s,a,r,s')$ the network currently estimates $\max_{a'}Q_\theta(s',a')=50$. Compute the target $y = r + \gamma\,(1-d)\max_{a'}Q_\theta(s',a')$ in three cases:

1. the pole falls at $s'$ (the episode **terminates**);
2. the episode is **truncated** at $s'$ because step 500 has been reached;
3. as in case 2, but the implementation wrongly sets $d=1$ whenever the episode ends (storing `done = terminated or truncated` in the `failure` field instead of `terminated`).

Then explain what the network learns in case 3, keeping in mind that the time step is *not* part of the Cart-Pole state, and compute the value of a state in which the pole is balanced forever.

**Step 1 — Terminated.** $s'$ is terminal, $d=1$: no future reward is possible.

$\displaystyle y = 1 + 0.99\,(1-1)\,50 = 1.$

**Step 2 — Truncated.** $s'$ is *not* terminal, $d=0$: the agent could have continued, so we bootstrap.

$\displaystyle y = 1 + 0.99\,(1-0)\,50 = 50.5.$

**Step 3 — Truncation treated as termination.** With $d=1$ the target is again $y=1$: the same state $s'$ is labelled as worth $50.5$ when reached at step 200 and as worth $1$ when reached at step 500. Since the time step is not observed, the two situations are **the same input** for the network, and the regression is asked to fit two contradictory labels. The effect is not confined to the last state: a well-balanced pole at step 500 looks exactly like a well-balanced pole at any other step, so the artificially low value spreads by generalisation to *all* good states, holding down the values the agent is trying to learn and slowing learning considerably.

**Step 4 — Value of a perfectly balanced state.** If the pole never falls the agent receives $r=1$ forever, and the discounted return is

$\displaystyle q_*(s,a) = \sum_{k=0}^{\infty} \gamma^{k} = \frac{1}{1-\gamma} = \frac{1}{0.01} = 100.$

This is the value the network should converge to for good states, and it is why the estimate $50$ in the exercise is plausible for a partially trained agent. The undiscounted return of an episode, instead, is capped at $500$ by the truncation: the *score* we plot and the *values* the network learns live on different scales.

**Key concept**

A **terminal** state has no future value ($d=1$); a **truncated** episode simply stops observing a state that does have future value ($d=0$). Confusing the two injects a label that depends on an unobservable quantity (time) and, through generalisation, corrupts the values of all similar states. In code, store `terminated` in the `failure` field, never `terminated or truncated` — the notes do this in every training loop of the chapter.

### Exercise 7.7 — Fixed targets vs. recomputed targets

Take again the scalar approximator of Exercise 7.5: $Q_\theta(s)=\theta\,\phi(s)$ with $\phi(s)=\phi(s')=1$, the single transition $s\to s'$ with $r=0$, $\gamma=0.9$, initial $\theta_0=1$, $\alpha=0.5$. The "batch" contains this one transition and is used for **two epochs** of optimisation.

1. **Fitted (NFQ) variant:** compute the target once from $\theta_0$, then perform two semi-gradient steps toward that *fixed* target.
2. **Moving-target variant:** recompute the target from the current $\theta$ before each of the two steps.
3. Compare the two results, and describe what happens to the loss $\mathcal{L}(\theta)=(y-Q_\theta)^2$ in the two variants as the number of epochs grows.

**Step 1 — Fixed target.** Computed once: $\ y = r + \gamma\,\theta_0 = 0.9$, and held constant.

$\displaystyle \text{epoch 1:}\quad \delta = 0.9 - 1 = -0.1, \qquad \theta_1 = 1 + 0.5(-0.1) = 0.95;$

$\displaystyle \text{epoch 2:}\quad \delta = 0.9 - 0.95 = -0.05, \qquad \theta_2 = 0.95 + 0.5(-0.05) = 0.925.$

The parameter moves toward $0.9$ and, with more epochs, converges exactly to it: $\theta_k - 0.9 = 0.5^{k}\,(\theta_0-0.9)$. The loss decreases monotonically to $0$, as in any well-posed regression.

**Step 2 — Recomputed target.**

$\displaystyle \text{epoch 1:}\quad y = 0.9\,\theta_0 = 0.9, \quad \delta = -0.1, \qquad \theta_1 = 0.95;$

$\displaystyle \text{epoch 2:}\quad y = 0.9\,\theta_1 = 0.855, \quad \delta = 0.855 - 0.95 = -0.095, \qquad \theta_2 = 0.95 + 0.5(-0.095) = 0.9025.$

Each epoch is now an iteration of Exercise 7.5: $\theta_k = 0.95^k\,\theta_0 \to 0$.

**Step 3 — Comparison.** After two epochs the two variants give $\theta_2 = 0.925$ and $\theta_2 = 0.9025$. In this deterministic toy problem both are moving in the right direction (the true value is $0$), and the moving-target variant is even faster, because it performs a Bellman backup at every epoch instead of every batch. The difference lies in *what the optimisation is doing*. With fixed targets each fit is a **well-posed supervised problem**: the loss has a fixed minimiser ($\theta = y$), it decreases monotonically, and reaching a small loss means the network has *fitted the targets*; the Bellman backup happens once per batch, when the targets are recomputed, and it is a contraction ($y_{\text{new}} = \gamma\,\theta$) exactly as in value iteration. With recomputed targets the loss has **no fixed minimiser**: the quantity being minimised changes at every step, so a small loss means only that $Q$ is close to the *current* target, not that the fit has converged, and with a neural network (many parameters, correlated samples, large steps) the target can move faster than the network follows it. This is why NFQ, like fitted Q iteration in general, freezes the targets for the duration of each fit: in `optimize_nfq` the targets are computed once, before the epoch loop.

**Key concept**

"Fitted" means: compute the targets once with the current network, then solve an ordinary regression toward these **fixed** labels; only then recompute the targets. Each fit is well-posed, and the Bellman backup is confined to the moments in which the targets are refreshed. DQN's target network takes the idea one step further by freezing the targets across *many* batches.

### Exercise 7.8 — The decaying $\varepsilon$ schedule

The DQN agent of the notes explores with a probability that decays **geometrically** over the first $K=10\,000$ environment steps, from $\varepsilon_{\max}=1$ to $\varepsilon_{\min}=0.01$, and then stays at $\varepsilon_{\min}$ (`np.logspace` in `DecayingEpsilonGreedyPolicy`):

$\displaystyle \varepsilon_k = \varepsilon_{\max}\left(\frac{\varepsilon_{\min}}{\varepsilon_{\max}}\right)^{k/(K-1)}, \qquad k = 0,1,\ldots,K-1.$

1. Compute $\varepsilon_k$ at $k = 2500$, $5000$ and $7500$.
2. After how many steps does the exploration probability fall below $0.1$?
3. Estimate the expected number of random actions taken during the decay, and compare it with a *linear* schedule between the same two values over the same number of steps.
4. On Cart-Pole, episodes last about 20 steps at the beginning of training and up to 500 steps once the agent can balance the pole. Why does the notebook count the schedule in steps rather than in episodes?

**Step 1 — Values.** With $\varepsilon_{\min}/\varepsilon_{\max} = 10^{-2}$ the schedule is $\varepsilon_k = 10^{-2k/(K-1)}$:

$\displaystyle \varepsilon_{2500} = 10^{-0.5} \approx 0.316, \qquad \varepsilon_{5000} = 10^{-1.0} \approx 0.100, \qquad \varepsilon_{7500} = 10^{-1.5} \approx 0.032.$

Each quarter of the schedule divides $\varepsilon$ by the same factor $\sqrt{10}\approx 3.16$: this is what "constant multiplicative decay" means.

**Step 2 — Crossing $0.1$.** $\ \varepsilon_k < 0.1 \iff 10^{-2k/(K-1)} < 10^{-1} \iff k > (K-1)/2 = 4999.5$, so from step $k = 5000$ onwards. Half of the schedule is spent above $0.1$ (mostly random behaviour) and half below it (mostly greedy behaviour).

**Step 3 — Expected number of random actions.** The expected number of random actions during the decay is $\sum_{k=0}^{K-1}\varepsilon_k$, which for a geometric sequence is well approximated by the integral:

$\displaystyle \sum_{k=0}^{K-1} \varepsilon_k \approx \int_0^{K-1} 10^{-2t/(K-1)}\,dt = \frac{K-1}{\ln 100}\,\big(1 - 10^{-2}\big) = \frac{9999 \times 0.99}{4.605} \approx 2150.$

(The exact sum is $\approx 2150$.) A linear schedule from $1$ to $0.01$ over the same $10\,000$ steps would take $\tfrac{1+0.01}{2}\times 10\,000 \approx 5050$ random actions: more than twice as many. The geometric schedule explores intensely for a short time and then hands control to the greedy policy quickly, which is what we want once the network knows something.

**Step 4 — Steps, not episodes.** A schedule in episodes would decay $\varepsilon$ by a fixed factor per episode. Early episodes last about $20$ steps, so $100$ episodes of the schedule would cover only $\approx 2000$ transitions, far too few to fill the replay memory with diverse experience before exploration is switched off; later, with $500$-step episodes, the same number of episodes would cover $50\,000$ transitions. Counting in *steps* ties exploration to the amount of experience actually collected, which is what the learning progress depends on, and makes the schedule independent of how long episodes happen to be.

**Key concept**

The $\varepsilon$ schedule is a hyperparameter like any other: it decides how much of the replay memory is filled with random experience and how soon the agent starts trusting its own estimates. A geometric decay in environment steps, with a floor $\varepsilon_{\min}>0$, is the standard DQN choice; the floor keeps some off-policy transitions arriving for the rest of training.

### Exercise 7.9 — Replay buffer and decorrelation

A replay buffer holds $N=100$ transitions; each optimisation step samples a mini-batch of $m=10$ **without replacement**, uniformly at random (`np.random.choice(..., replace=False)` in `sample_experiences`).

1. What is the probability that a specific transition is included in a given mini-batch?
2. Two *temporally consecutive* transitions (collected at steps $t$ and $t{+}1$) are strongly correlated. What is the probability that **both** land in the same mini-batch?
3. Interpret why this addresses the IID-violation problem of online updates.

**Step 1 — Inclusion probability.** With uniform sampling without replacement, each transition is equally likely to be among the $m$ chosen out of $N$:

$\displaystyle P(\text{a specific transition in the batch}) = \frac{m}{N} = \frac{10}{100} = 0.1.$

**Step 2 — Both correlated samples together.** The first of the pair is included with probability $m/N$; given that, the second occupies one of the remaining $m-1$ of the remaining $N-1$ slots:

$\displaystyle P(\text{both}) = \frac{m}{N}\cdot\frac{m-1}{N-1} = \frac{10}{100}\cdot\frac{9}{99} = 0.0091\ (\approx 0.9\%).$

**Step 3 — Interpretation.** Online updates use consecutive, highly correlated transitions every step, badly violating the IID assumption of stochastic-gradient training. Sampling from a large buffer makes it *unlikely* that adjacent, correlated transitions are learned together (here under $1\%$), so each mini-batch mixes experiences from many different times and policies — approximately IID. With the notebook's buffers of $20\,000$ to $100\,000$ transitions and mini-batches of $32$ to $128$, the probability is smaller still.

**Key concept**

Experience replay converts a correlated online stream into a near-IID dataset by **breaking temporal adjacency**, which is what lets standard gradient-based optimisation remain stable. It also lets each transition be reused many times, improving sample efficiency — and it is only legitimate because Q-learning is off-policy.

### Exercise 7.10 — DQN vs. Double DQN targets

For a sampled transition with reward $r=1$ and $\gamma=0.9$, the next state $s'$ has action-values

$\displaystyle Q_{\theta_{\text{online}}}(s',\cdot)=(1.0,\ 3.0), \qquad Q_{\theta_{\text{target}}}(s',\cdot)=(2.5,\ 2.0).$

Compute the **DQN** target and the **Double DQN** target, and explain why DDQN reduces overestimation.

**Step 1 — DQN target.** DQN takes the **max over the target network** (selection *and* evaluation by $\theta_{\text{target}}$):

$\displaystyle y_{\text{DQN}} = r + \gamma\max_{a'}Q_{\theta_{\text{target}}}(s',a') = 1 + 0.9\max(2.5,2.0) = 1 + 0.9(2.5) = 3.25.$

**Step 2 — Double DQN target.** DDQN **selects** the action with the online network and **evaluates** it with the target network:

$\displaystyle a^* = \arg\max_{a'}Q_{\theta_{\text{online}}}(s',a') = a_2 \ (\text{online value }3.0),$
$\displaystyle y_{\text{DDQN}} = r + \gamma\,Q_{\theta_{\text{target}}}(s',a^*) = 1 + 0.9\,Q_{\theta_{\text{target}}}(s',a_2) = 1 + 0.9(2.0) = 2.8.$

In `optimize_ddqn` these are the two lines `best_actions = online_q(next_states).argmax(dim=1)` and `q_next = target_q(next_states).gather(1, ...)`.

**Step 3 — Why lower.** The online argmax is $a_2$, but the *target* network rates $a_2$ at only $2.0$ (not the $2.5$ that DQN's own max would grab from $a_1$). By forcing the evaluation to come from a different set of estimates, DDQN avoids automatically pairing "the action that looks best" with "the value that looks biggest", so $y_{\text{DDQN}}=2.8 < 3.25 = y_{\text{DQN}}$. Note that DDQN does not always give the lower target: when both networks agree on the best action the two targets coincide, and the effect is statistical, not pointwise.

**Key concept**

DQN's single max both *selects* and *evaluates* with the same (noisy) estimates, producing the maximisation bias of Chapter 05 — now in a deep setting. DDQN decouples the two using the target network the algorithm already maintains, at no extra cost. The chapter shows the benefit on Lunar Lander, whose four actions and $\pm 100$ terminal rewards make the value estimates noisy and large; on Cart-Pole, with two actions and a constant reward, there would be little for it to exploit.

### Exercise 7.11 — Prioritised Experience Replay

Four transitions in the buffer have TD errors $\delta=(2.0,\,1.0,\,0.5,\,0.0)$. Prioritised replay uses priority $p_i=|\delta_i|+\epsilon_p$ with $\epsilon_p=0.5$, sampling exponent $\alpha=1$, and importance-sampling exponent $\beta=0.5$ (buffer size $N=4$).

1. Compute the priorities and the sampling probabilities $P(i)=p_i^{\alpha}/\sum_k p_k^{\alpha}$.
2. Compute the importance-sampling weights $w_i=\big(\tfrac1N\tfrac1{P(i)}\big)^{\beta}$, normalised so the largest is $1$.
3. Explain the role of $\epsilon_p$ and of the IS weights.
4. A fifth transition is now stored. With what priority does it enter the memory, and what is its sampling probability?

**Step 1 — Priorities and probabilities.** $p_i=|\delta_i|+0.5 = (2.5,\,1.5,\,1,\,0.5)$, sum $=5.5$. With $\alpha=1$:

$\displaystyle P = \Big(0.4545,\ 0.2727,\ 0.1818,\ 0.0909\Big).$

**Step 2 — Importance-sampling weights.** $w_i=\big(N\,P(i)\big)^{-\beta}$ with $N=4,\ \beta=0.5$:

$\displaystyle w = (0.7416,\,0.9574,\,1.1726,\,1.6583)\ \xrightarrow{\div\max} \ \tilde w = (0.4472,\,0.5774,\,0.7071,\,1).$

In `sample_prioritized_experiences` this is `weights = (entries * probabilities[indices]) ** (-beta)` followed by `weights / weights.max()`.

**Step 3 — Interpretation.** The offset $\epsilon_p$ keeps the zero-TD-error transition (transition 4) at a **non-zero** priority ($P=0.0909>0$), so it can still be replayed — without it, a transition that once had $\delta=0$ would never be sampled again, even though the network, and therefore its TD error, keep changing. The IS weights **counteract the sampling bias**: the most-sampled (highest-priority) transition 1 gets the *smallest* weight $0.4472$, while the least-sampled transition 4 gets weight $1$, so that frequently-replayed transitions do not dominate the gradient. The weights multiply the squared errors in the loss, `(weights * td_errors.pow(2)).mean()`; they do not touch the priorities.

**Step 4 — A new transition.** Its TD error has not been measured, so `store_prioritized_experience` gives it the **largest priority currently stored**, $p_5 = 2.5$. The sum of priorities becomes $5.5 + 2.5 = 8.0$ and

$\displaystyle P(5) = \frac{2.5}{8.0} = 0.3125,$

tied with transition 1 for the highest probability. This makes it *likely*, not certain, that the new transition is replayed soon; once it has been sampled its priority is replaced by its measured $|\delta|+\epsilon_p$.

**Key concept**

PER replays *surprising* (high-$|\delta|$) transitions more often to learn faster, but this skews the data distribution; importance-sampling weights re-balance the updates so the expected gradient stays (approximately) correct. $\alpha$ tunes how greedy the prioritisation is; $\beta$ tunes how fully the bias is corrected. The notebook's experiments use $\alpha = 0.6$, a fixed $\beta = 0.4$ and $\epsilon_p = 0.01$; the original paper anneals $\beta$ towards $1$.

### Exercise 7.12 — Value scales on Acrobot, and why PER helps there

On Acrobot every step gives $r=-1$ until the tip reaches the target height; the transition that reaches it gives $r=0$ and terminates the episode. Take $\gamma=0.99$.

1. Under a policy that reaches the goal in exactly $k$ steps from state $s$, compute the action value $q(s,a)$ of the first action, and evaluate it for $k=10$, $100$ and $500$. What is the value of a state from which the goal is never reached?
2. Early in training the network predicts $Q_\theta \approx -60$ for every state and action. Compute the TD error of an ordinary transition (next state also predicted at $-60$) and of a transition that reaches the goal.
3. With $\epsilon_p=0.01$ and $\alpha=0.6$, how many times more likely is the goal-reaching transition to be sampled than a given ordinary one? Relate this to the comparison of uniform and prioritised replay in the notes.

**Step 1 — Value of a state $k$ steps from the goal.** The agent takes $k$ actions: the first $k-1$ transitions give $-1$ and the $k$-th, which reaches the goal, gives $0$, so

$\displaystyle q(s,a) = -\sum_{j=0}^{k-2}\gamma^{j} = -\frac{1-\gamma^{k-1}}{1-\gamma},$

which gives $q = -8.65$ for $k=10$, $-63.0$ for $k=100$ and $-99.3$ for $k=500$. If the goal is never reached the return is $-\sum_{j\ge 0}\gamma^j = -1/(1-\gamma) = -100$. All action values therefore lie in $[-100, 0]$: the discounted value of "never getting there" is $-100$, even though the undiscounted return of a truncated episode is $-500$. (The evaluation curves of the notes plot the undiscounted return, which is why they go down to $-500$ and up to about $-90$.)

**Step 2 — TD errors early in training.** With every prediction at $-60$:

$\displaystyle \text{ordinary transition:}\quad \delta = r + \gamma Q_\theta(s',a^*) - Q_\theta(s,a) = -1 + 0.99(-60) - (-60) = -1 - 59.4 + 60 = -0.4,$

$\displaystyle \text{goal-reaching transition } (d=1):\quad \delta = 0 + 0 - (-60) = +60.$

The thousands of "swinging around" transitions carry almost no information for the network as it stands; the one that ends the episode contradicts the current estimates by sixty units.

**Step 3 — Sampling ratio.** Priorities: $p_{\text{goal}} = 60 + 0.01 = 60.01$, $p_{\text{ordinary}} = 0.4 + 0.01 = 0.41$. With $\alpha = 0.6$ the sampling probabilities are proportional to $p^{0.6}$, so

$\displaystyle \frac{P(\text{goal})}{P(\text{ordinary})} = \left(\frac{60.01}{0.41}\right)^{0.6} = 146.4^{\,0.6} \approx 19.9.$

The goal-reaching transition is sampled about $20$ times more often than any single ordinary one (about $146$ times with $\alpha=1$, once with uniform replay). This is the mechanism behind the Acrobot experiment of the notes: the informative transitions are rare, and prioritised replay concentrates the updates on them, so the correct value propagates backwards from the goal sooner and the mean return rises earlier. The exponent $\alpha < 1$ keeps the ordinary transitions in play, which matters because once the terminal transition has been fitted its TD error, and its priority, collapse.

**Key concept**

Discounting bounds the values an agent can learn ($|q| \le 1/(1-\gamma)$ for unit rewards) regardless of episode length, and a sparse reward makes almost all TD errors alike except at the few transitions that carry the signal. PER is designed for exactly this situation; on a task with shaped rewards and uniform TD errors it has little to exploit.

### Exercise 7.13 — Dueling architecture and identifiability

A dueling network outputs a state-value $V(s)=5$ and raw advantages $A(s,\cdot)=(1,\,-1,\,2)$ for three actions. It combines them with the **mean-subtracted** aggregation

$\displaystyle Q(s,a) = V(s) + \Big(A(s,a) - \tfrac{1}{|\mathcal{A}|}\sum_{a'}A(s,a')\Big).$

1. Compute $Q(s,\cdot)$ and verify that $V(s)=\tfrac1{|\mathcal{A}|}\sum_a Q(s,a)$.
2. Show that the *naive* aggregation $Q=V+A$ is **unidentifiable**, and explain how the mean-subtraction fixes it.
3. The notes test the dueling network on Pendulum with $101$ discretised torques. Why is a large number of similar actions the setting in which the split is expected to help?

**Step 1 — Aggregate.** Mean advantage $\bar A=\tfrac{1+(-1)+2}{3}=0.6667$. Then

$\displaystyle Q(s,\cdot) = 5 + \big[(1,-1,2)-0.6667\big] = (5.3333, 3.3333, 6.3333).$

Check: $\tfrac13\sum_a Q(s,a) = \tfrac{5.3333+3.3333+6.3333}{3} = 5 = V(s)$. ✓ The mean-subtraction forces the value head to equal the mean action-value, which is the arithmetic mean of the network's $Q$-values and not necessarily $v_\pi(s)$; the ranking of the actions is unchanged.

**Step 2 — Unidentifiability of $Q=V+A$.** The same $Q(s,\cdot)=(5.3333, 3.3333, 6.3333)$ can be produced by infinitely many $(V,A)$ splits, e.g.

- $V=5,\ A=(0.3333,-1.6667,1.3333)$ (mean $0$), or
- $V=0,\ A=(5.3333,3.3333,6.3333)$ (mean $5$),

since only the *sum* $V+A$ enters $Q$. Given only $Q$, we cannot recover $V$ and $A$ separately — the decomposition is not unique, and the two heads can drift apart by a constant without the loss noticing. Imposing $\sum_a A(s,a)=0$ (equivalently, subtracting the mean advantage, as `DuelingNetwork.forward` does with `advantages - advantages.mean(dim=-1, keepdim=True)`) removes this freedom: it pins $V(s)=\tfrac1{|\mathcal{A}|}\sum_a Q(s,a)$ and $A(s,a)=Q(s,a)-V(s)$, a *unique* decomposition.

**Step 3 — Many similar actions.** On Pendulum neighbouring torques have almost the same effect, so for a given state the $101$ action values differ by small amounts around a common level set by the pendulum's orientation and speed. A standard network must learn $101$ nearly identical numbers per state, and a transition updates the output weights of only the action that was tried (`gather` selects it; the shared hidden layers do receive gradient). In the dueling network every transition updates the shared value head, which carries most of what there is to learn about the state, while the advantage head only has to represent the small differences; information gathered with one torque therefore improves the estimates of all the others. This is why the notes see a faster rise on Pendulum; with two actions, as on Cart-Pole, there would be almost nothing to share.

**Key concept**

The dueling network separately estimates "how good is this state" ($V$) and "how much better is each action" ($A$), sharing features so that learning about one action informs the state value. The mean-subtraction is not cosmetic — it is what makes the two heads *identifiable* and stably trainable. The gain depends on the action space: the more actions share a common value, the more there is to gain.